# Phase 3 — Combined resource scores

Joins solar + wind screening layers for downstream Phase 4+ (yield, seasonal, composite S).

**All logic runs in this notebook** (no external script imports).

| Input | Output |
|-------|--------|
| `solar_scores_by_cell.parquet` (`07`) | `cell_resource_scores.parquet` |
| `wind_scores_by_cell.parquet` (`08`) | `hybrid_screening_score` |

In [1]:
from pathlib import Path

import polars as pl

ROOT = Path('../..').resolve()
SOLAR = ROOT / 'data' / 'processed' / 'solar_scores_by_cell.parquet'
WIND = ROOT / 'data' / 'processed' / 'wind_scores_by_cell.parquet'
OUT = ROOT / 'data' / 'processed' / 'cell_resource_scores.parquet'
JOIN = ['latitude', 'longitude']

In [2]:
for p in (SOLAR, WIND):
    if not p.exists():
        raise FileNotFoundError(f'Missing {p} — run 07 and 08 first')

score_cols = [
    'resource_observed', 'resource_observed_pct', 'resource_land_adj_oof',
    'resource_opportunity_oof', 'resource_screening_score',
]

solar = pl.read_parquet(SOLAR)
wind = pl.read_parquet(WIND)
solar = solar.select(JOIN + [c for c in score_cols if c in solar.columns])
wind = wind.select(JOIN + score_cols)

solar = solar.rename({c: f'solar_{c}' for c in score_cols if c in solar.columns})
wind = wind.rename({c: f'wind_{c}' for c in score_cols})

combined = solar.join(wind, on=JOIN, how='inner')
combined = combined.with_columns(
    ((pl.col('solar_resource_screening_score') + pl.col('wind_resource_screening_score')) / 2)
    .alias('hybrid_screening_score')
)

combined.write_parquet(OUT)
print(f'Wrote {len(combined):,} rows -> {OUT}')
combined.head()

Wrote 1,089 rows -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/cell_resource_scores.parquet


latitude,longitude,solar_resource_observed,solar_resource_observed_pct,solar_resource_land_adj_oof,solar_resource_opportunity_oof,solar_resource_screening_score,wind_resource_observed,wind_resource_observed_pct,wind_resource_land_adj_oof,wind_resource_opportunity_oof,wind_resource_screening_score,hybrid_screening_score
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
42.39,-79.26,307.224073,0.963728,300.555176,6.668897,0.963728,85.120093,0.883838,82.196167,2.923926,0.883838,0.923783
41.94,-73.77,305.184693,0.923783,306.734253,-1.54956,0.923783,47.851187,0.147842,47.124653,0.726534,0.147842,0.535813
43.11,-73.86,298.392867,0.711662,293.347595,5.045272,0.711662,45.042919,0.100092,44.452396,0.590522,0.100092,0.405877
42.21,-78.18,297.999296,0.684114,297.596924,0.402372,0.684114,73.138325,0.705234,72.110817,1.027508,0.705234,0.694674
42.12,-77.91,299.050447,0.75023,298.94577,0.104677,0.75023,73.358383,0.708907,58.965748,14.392635,0.708907,0.729568
